# Comparacao de Algoritmos de Ordenacao

Este notebook compara o desempenho de tres algoritmos de ordenacao:

- Insertion Sort
- Merge Sort
- Quick Sort

A comparacao segue os mesmos moldes do notebook de forca bruta: validacao de corretude, benchmark por cenario/tamanho, consolidacao em tabelas e visualizacoes.

## 1. Setup do Colab e bibliotecas

In [ ]:
from time import perf_counter
import random
from statistics import mean, stdev

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

## 2. Implementacao dos algoritmos (Insertion, Merge, Quick)

No Quick Sort, a estrategia de pivo usada e o elemento central da lista para reduzir degeneracao em entradas parcialmente ordenadas.

In [ ]:
def insertion_sort(arr):
    a = arr.copy()
    for i in range(1, len(a)):
        chave = a[i]
        j = i - 1
        while j >= 0 and a[j] > chave:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = chave
    return a


def merge_sort(arr):
    if len(arr) <= 1:
        return arr.copy()

    meio = len(arr) // 2
    esquerda = merge_sort(arr[:meio])
    direita = merge_sort(arr[meio:])

    resultado = []
    i = j = 0

    while i < len(esquerda) and j < len(direita):
        if esquerda[i] <= direita[j]:
            resultado.append(esquerda[i])
            i += 1
        else:
            resultado.append(direita[j])
            j += 1

    resultado.extend(esquerda[i:])
    resultado.extend(direita[j:])
    return resultado


def quick_sort(arr):
    if len(arr) <= 1:
        return arr.copy()

    pivo = arr[len(arr) // 2]
    menores = [x for x in arr if x < pivo]
    iguais = [x for x in arr if x == pivo]
    maiores = [x for x in arr if x > pivo]

    return quick_sort(menores) + iguais + quick_sort(maiores)


algoritmos = {
    "Insertion Sort": insertion_sort,
    "Merge Sort": merge_sort,
    "Quick Sort": quick_sort,
}

## 3. Validacao de corretude das ordenacoes

In [ ]:
casos_teste = {
    "vazio": [],
    "unico": [7],
    "duplicados": [3, 1, 2, 3, 2, 1],
    "ordenado": [1, 2, 3, 4, 5],
    "reverso": [5, 4, 3, 2, 1],
    "misto": [9, -1, 4, 0, 4, 2],
}

for nome_caso, arr in casos_teste.items():
    esperado = sorted(arr)
    for nome_alg, func in algoritmos.items():
        obtido = func(arr)
        assert obtido == esperado, (
            f"Falha em {nome_alg} no caso {nome_caso}: {obtido} != {esperado}"
        )

print("Todos os testes de corretude passaram.")

## 4. Gerador de entradas para benchmark

In [ ]:
def gerar_entrada(n, cenario):
    if cenario == "aleatorio":
        return random.sample(range(10 * n), n)
    if cenario == "crescente":
        return list(range(n))
    if cenario == "decrescente":
        return list(range(n, 0, -1))
    raise ValueError(f"Cenario desconhecido: {cenario}")


def gerar_casos(n, cenarios=("aleatorio", "crescente", "decrescente")):
    return {cenario: gerar_entrada(n, cenario) for cenario in cenarios}

## 5. Funcao de medicao de tempo e operacao basica (comparacoes)

In [ ]:
class ContadorComparacoes:
    def __init__(self):
        self.total = 0


class ValorComparavel:
    def __init__(self, valor, contador):
        self.valor = valor
        self.contador = contador

    def _conta(self):
        self.contador.total += 1

    def __lt__(self, other):
        self._conta()
        return self.valor < other.valor

    def __le__(self, other):
        self._conta()
        return self.valor <= other.valor

    def __gt__(self, other):
        self._conta()
        return self.valor > other.valor

    def __eq__(self, other):
        self._conta()
        return self.valor == other.valor


def medir_comparacoes(algoritmo, arr):
    contador = ContadorComparacoes()
    entrada_contavel = [ValorComparavel(x, contador) for x in arr]
    saida = algoritmo(entrada_contavel)
    saida_valores = [x.valor for x in saida]

    assert saida_valores == sorted(arr), f"Erro de ordenacao em {algoritmo.__name__}"
    return contador.total


def benchmark_tempo(algoritmo, arr, repeticoes=5, aquecimento=True):
    if aquecimento:
        _ = algoritmo(arr.copy())

    tempos = []
    for _ in range(repeticoes):
        entrada = arr.copy()
        inicio = perf_counter()
        saida = algoritmo(entrada)
        fim = perf_counter()

        assert saida == sorted(arr), f"Erro de ordenacao em {algoritmo.__name__}"
        tempos.append(fim - inicio)

    media = mean(tempos)
    desvio = stdev(tempos) if len(tempos) > 1 else 0.0
    return media, desvio

## 6. Execucao dos experimentos por tamanho e cenario

In [ ]:
tamanhos = [100, 500, 1000, 2000, 5000, 10000]
cenarios = ["aleatorio", "crescente", "decrescente"]
repeticoes = 7

registros = []

for n in tamanhos:
    casos = gerar_casos(n, cenarios)
    for cenario, dados_base in casos.items():
        for nome_alg, func in algoritmos.items():
            tempo_medio, tempo_std = benchmark_tempo(
                func, dados_base, repeticoes=repeticoes, aquecimento=True
            )
            comparacoes = medir_comparacoes(func, dados_base)

            registros.append(
                {
                    "algoritmo": nome_alg,
                    "cenario": cenario,
                    "tamanho": n,
                    "tempo_medio": tempo_medio,
                    "tempo_std": tempo_std,
                    "comparacoes": comparacoes,
                }
            )

print(f"Total de registros: {len(registros)}")

## 7. Consolidacao dos resultados em DataFrame

In [ ]:
df = pd.DataFrame(registros)
df = df.sort_values(["cenario", "tamanho", "tempo_medio"]).reset_index(drop=True)

display(df.head(12))

pivot_tempo = df.pivot_table(
    index=["cenario", "tamanho"],
    columns="algoritmo",
    values="tempo_medio",
    aggfunc="first",
)

pivot_std = df.pivot_table(
    index=["cenario", "tamanho"],
    columns="algoritmo",
    values="tempo_std",
    aggfunc="first",
)

pivot_comparacoes = df.pivot_table(
    index=["cenario", "tamanho"],
    columns="algoritmo",
    values="comparacoes",
    aggfunc="first",
)

print("Tabela pivo de tempo medio (s):")
display(pivot_tempo)
print("Tabela pivo de desvio padrao (s):")
display(pivot_std)
print("Tabela pivo de comparacoes:")
display(pivot_comparacoes)

## 8. Visualizacao comparativa de desempenho

In [ ]:
for cenario in cenarios:
    parte = df[df["cenario"] == cenario]

    fig, ax = plt.subplots()
    for nome_alg in algoritmos.keys():
        dados_alg = parte[parte["algoritmo"] == nome_alg]
        ax.plot(dados_alg["tamanho"], dados_alg["tempo_medio"], marker="o", label=nome_alg)

    ax.set_title(f"Tempo medio por tamanho | cenario: {cenario}")
    ax.set_xlabel("Tamanho da lista")
    ax.set_ylabel("Tempo medio (s)")
    ax.set_yscale("log")
    ax.legend()
    plt.show()

for cenario in cenarios:
    parte = df[df["cenario"] == cenario]

    fig, ax = plt.subplots()
    for nome_alg in algoritmos.keys():
        dados_alg = parte[parte["algoritmo"] == nome_alg]
        ax.plot(dados_alg["tamanho"], dados_alg["comparacoes"], marker="o", label=nome_alg)

    ax.set_title(f"Comparacoes por tamanho | cenario: {cenario}")
    ax.set_xlabel("Tamanho da lista")
    ax.set_ylabel("Numero de comparacoes")
    ax.set_yscale("log")
    ax.legend()
    plt.show()

## 9. Resumo quantitativo (medias, desvio, ranking e speedup)

In [ ]:
resumo = df.copy()
resumo["ranking_tempo"] = resumo.groupby(["cenario", "tamanho"])["tempo_medio"].rank(method="dense")
resumo["ranking_comparacoes"] = resumo.groupby(["cenario", "tamanho"])["comparacoes"].rank(method="dense")

base_insertion = (
    resumo[resumo["algoritmo"] == "Insertion Sort"]
    .set_index(["cenario", "tamanho"])["tempo_medio"]
)
base_insertion_comp = (
    resumo[resumo["algoritmo"] == "Insertion Sort"]
    .set_index(["cenario", "tamanho"])["comparacoes"]
)

idx = resumo.set_index(["cenario", "tamanho"]).index
resumo["tempo_insertion"] = idx.map(base_insertion)
resumo["comparacoes_insertion"] = idx.map(base_insertion_comp)

resumo["speedup_vs_insertion"] = resumo["tempo_insertion"] / resumo["tempo_medio"]
resumo["reducao_comp_vs_insertion"] = resumo["comparacoes_insertion"] / resumo["comparacoes"]

colunas = [
    "algoritmo",
    "cenario",
    "tamanho",
    "tempo_medio",
    "tempo_std",
    "comparacoes",
    "ranking_tempo",
    "ranking_comparacoes",
    "speedup_vs_insertion",
    "reducao_comp_vs_insertion",
]

resumo_final = resumo[colunas].sort_values(
    ["cenario", "tamanho", "ranking_tempo", "ranking_comparacoes", "tempo_medio"]
)

display(resumo_final)

agregado = (
    resumo_final.groupby("algoritmo", as_index=False)
    .agg(
        tempo_medio_global=("tempo_medio", "mean"),
        tempo_std_global=("tempo_medio", "std"),
        comparacoes_medias=("comparacoes", "mean"),
        ranking_medio_tempo=("ranking_tempo", "mean"),
        ranking_medio_comparacoes=("ranking_comparacoes", "mean"),
        speedup_medio_vs_insertion=("speedup_vs_insertion", "mean"),
        reducao_media_comp_vs_insertion=("reducao_comp_vs_insertion", "mean"),
    )
    .sort_values(["ranking_medio_tempo", "ranking_medio_comparacoes"])
)

print("Resumo agregado por algoritmo:")
display(agregado)